# Necessary Packages 

In [0]:
from pyspark.sql.functions import *

# Fetching All Data

In [0]:
exchange_rates_df = spark.table("01_dev_bronze.raw.exchange_rates")
customers_df=spark.table("01_dev_bronze.raw.customers")
order_items_df = spark.table("01_dev_bronze.raw.order_items")
orders_df = spark.table("01_dev_bronze.raw.orders")
products_df = spark.table("01_dev_bronze.raw.products")

# Transforming Exchange Rates Table

In [0]:
exchange_rates_df = exchange_rates_df.withColumn("exchange_rate_to_usd",col("exchange_rate_to_usd").cast("double"))

# Transforming Products Table

In [0]:
products_df=products_df.withColumn("product_name",trim(col("product_name")))\
    .withColumn("category",trim(col("category")))\
    .withColumn("price",col("price").cast("double"))

products_df=products_df.replace(["NULL","\\N","?",""],None)

products_df = products_df.join(exchange_rates_df.select("currency_code", "exchange_rate_to_usd"),
                           products_df.currency == exchange_rates_df.currency_code,
                           "left")
products_df = products_df.withColumn("base_price",round(col("price") * col("exchange_rate_to_usd"), 2).cast("double"))

products_df=products_df.withColumn("product_name",
                   when(
          col("product_name").isNull() , concat(
              col("category"),
              lit(' Product '),
              trim(regexp_replace(substring(col("product_id"),5,5), r'0', '')))
          ).otherwise(col("product_name")))

products_df = products_df.drop("currency_code")

# Transforming Orders Table

In [0]:
orders_df=orders_df.withColumn("customer_id",trim(col("customer_id")))\
    .withColumn("order_status",trim(col("order_status")))


orders_df=orders_df.replace(["NULL","\\N","?","" ],None)

orders_df = orders_df.withColumn(
    "order_date_cleaned",
    col("order_date").cast("string")
).withColumn(
    "total_base_amount",
    round(col("total_amount").cast("double"),2)
).drop("total_amount")

orders_df = orders_df.withColumn(
    "order_date_clean",
    coalesce(
        expr("try_to_date(order_date_cleaned, 'yyyy-MM-dd')"),
        expr("try_to_date(order_date_cleaned, 'yyyy/MM/dd')"),
        expr("try_to_date(order_date_cleaned, 'dd/MM/yyyy')"),
        expr("try_to_date(order_date_cleaned, 'dd-MM-yyyy')")
    )
)

orders_df = orders_df.withColumn("order_status_clean",
when((col("country_code") == "CN") & (col("order_status") == "已完成"), "COMPLETED")
.when((col("country_code") == "CN") & (col("order_status") == "待处理"), "PENDING")
.when((col("country_code") == "CN") & (col("order_status") == "已发货"), "SHIPPED")
.when((col("country_code") == "CN") & (col("order_status") == "已取消"), "CANCELLED")
.when((col("country_code") == "ES") & (col("order_status") == "completado"), "COMPLETED")
.when((col("country_code") == "ES") & (col("order_status") == "pendiente"), "PENDING")
.when((col("country_code") == "ES") & (col("order_status") == "enviado"), "SHIPPED")
.when((col("country_code") == "ES") & (col("order_status") == "cancelado"), "CANCELLED")
.when((col("country_code") == "DE") & (col("order_status") == "abgeschlossen"), "COMPLETED")
.when((col("country_code") == "DE") & (col("order_status") == "ausstehend"), "PENDING")
.when((col("country_code") == "DE") & (col("order_status") == "versandt"), "SHIPPED")
.when((col("country_code") == "DE") & (col("order_status") == "storniert"), "CANCELLED")
.when((col("country_code") == "IN") & (col("order_status") == "पूर्ण"), "COMPLETED")
.when((col("country_code") == "IN") & (col("order_status") == "लंबित"), "PENDING")
.when((col("country_code") == "IN") & (col("order_status") == "भेज दिया"), "SHIPPED")
.when((col("country_code") == "IN") & (col("order_status") == "रद्द"), "CANCELLED")
.when(col("order_status")=="completed", "COMPLETED")
.when(col("order_status")=="pending", "PENDING")
.when(col("order_status")=="shipped", "SHIPPED")
.when(col("order_status")=="cancelled", "CANCELLED")
.otherwise("CANCELLED"))

orders_df = orders_df.drop("order_date","order_date_cleaned").withColumnRenamed("order_date_clean", "order_date")
orders_df = orders_df.drop("order_status").withColumnRenamed("order_status_clean", "order_status")
orders_df = orders_df.drop("currency_code")

# Transforming Customers Table

In [0]:
customers_df=customers_df.withColumn("customer_name",trim(col("customer_name")))\
    .withColumn("email",trim(col("email")))

customers_df=customers_df.replace(["NULL","\\N","?",""],None)

customers_df = customers_df.withColumn(
    "registration_date_cleaned",
    col("registration_date").cast("string")
)

customers_df = customers_df.withColumn("registration_date_clean",
    coalesce(try_to_date(col("registration_date_cleaned"), "yyyy-MM-dd"),
             try_to_date(col("registration_date_cleaned"), "yyyy/MM/dd"),
             try_to_date(col("registration_date_cleaned"), "dd/MM/yyyy"),
             try_to_date(col("registration_date_cleaned"), "dd-MM-yyyy"))
)

customers_df = customers_df.withColumnRenamed("email", "email_address").withColumnRenamed("country_code", "country")
customers_df=customers_df.withColumn("country",
                                     when(col("country")=="UK", "United Kingdom")
.when(col("country")=="DE", "Germany")
.when(col("country")=="ES", "Spain")
.when(col("country")=="IN", "India")
.when(col("country")=="CN", "China")
.otherwise(col("country")))
customers_df=customers_df.withColumn("customer_name",
                   when(
          col("customer_name").isNull() , concat(lit('Customer '),
                                                 trim(regexp_replace(substring(col("customer_id"),5,5), r'0', '')))
    ).otherwise(col("customer_name")))

customers_df = customers_df.drop("registration_date","registration_date_cleaned").withColumnRenamed("registration_date_clean", "registration_date")

# Transforming Orders Items Table

In [0]:
order_items_df=order_items_df.withColumn("product_id",trim(col("product_id")))
order_items_df=order_items_df.replace(["NULL","\\N","?",""],None)

order_items_df = order_items_df.join(products_df.select(col("product_id").alias("products_product_id"), "currency"),
                           order_items_df.product_id == col("products_product_id"),
                           "left").drop("products_product_id")

order_items_df = order_items_df.join(exchange_rates_df.select("currency_code", "exchange_rate_to_usd"),
                           order_items_df.currency == exchange_rates_df.currency_code,
                           "left")

order_items_df = order_items_df.withColumn("base_unit_price",round(col("unit_price").cast("double") * col("exchange_rate_to_usd"), 2).cast("double"))\
                         .withColumn("base_line_total",round(col("line_total").cast("double") * col("exchange_rate_to_usd"), 2).cast("double"))

order_items_df = order_items_df.withColumn("quantity", col("quantity").cast("int"))
order_items_df = order_items_df.withColumn("unit_price", col("unit_price").cast("double"))
order_items_df = order_items_df.withColumn("line_total", col("line_total").cast("double"))

order_items_df = order_items_df.drop("currency_code")


# Saving All Tables

In [0]:
table_names={
    "02_dev_silver.transformed.customers": customers_df,
    "02_dev_silver.transformed.products": products_df,
    "02_dev_silver.transformed.order_items": order_items_df,
    "02_dev_silver.transformed.orders": orders_df,
    "02_dev_silver.transformed.exchange_rates": exchange_rates_df,
}

for table_name, df in table_names.items():
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)
